# FIT3181/5215 Kaggle Ensemble

This notebook is the executable record for the submitted six-member ensemble. It provides the GPU fine-tuning commands, records validation traces from the stored fine-tuned outputs, and rebuilds the submitted CSV.

## 1. Configure Training Environment

In [1]:
from pathlib import Path
import os
import random
import subprocess
import sys
import time

import numpy as np
import pandas as pd

ROOT = Path.cwd()
PROBS_DIR = ROOT / "probs"
SOURCE_DIR = ROOT / "src"
OUTPUT_DIR = ROOT / "notebook_outputs"
SEED = 1234

random.seed(SEED)
np.random.seed(SEED)

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    torch = None
    DEVICE = "cpu"

print(f"root: {ROOT}")
print(f"device: {DEVICE}")
print(f"probability arrays: {PROBS_DIR.exists()}")
print(f"source scripts: {SOURCE_DIR.exists()}")

root: /Users/harveyhoulahan/Desktop/Projects/maincode/kagglerun/33970076_assignment01_report
device: cpu
probability arrays: True
source scripts: True


## 2. Load and Prepare the Dataset

The training code expects `FIT5215_Dataset/` and `data/test_set/` at the project root. They are supplied separately when full fine-tuning is required. The stored probability arrays included in this submission support the executed validation and reconstruction cells below.

In [2]:
DATA_DIR = ROOT / "FIT5215_Dataset"
TEST_DIR = ROOT / "data" / "test_set"
DATA_AVAILABLE = DATA_DIR.exists() and TEST_DIR.exists()

print(f"training data available: {DATA_DIR.exists()}")
print(f"test data available: {TEST_DIR.exists()}")
print(f"full training enabled: {DATA_AVAILABLE and DEVICE == 'cuda'}")

training data available: False
test data available: False
full training enabled: False


## 3. Define the Model and Training Configuration

The submission is a weighted probability ensemble. Each command below writes a member probability file after fine-tuning and WiSE-FT interpolation.

In [3]:
MEMBERS = {
    "f1peL_a07": 2,
    "k2meta_a07": 2,
    "v3metaT_a07": 2,
    "d1dino_a10": 3,
    "g1eva448_a07": 2,
    "f2sig2_a07": 1,
}

TRAINING_COMMANDS = [
    "python src/train.py --tag f1peL --model PE-Core-L-14-336 --pretrained meta --res 336 --epochs 7 --lr 4e-6 --alphas 0.7",
    "python src/train.py --tag k2meta --model ViT-H-14-quickgelu --pretrained metaclip_fullcc --epochs 5 --lr 4e-6 --alphas 0.7",
    "python src/train.py --tag v3metaT --model ViT-H-14-quickgelu --pretrained metaclip_fullcc --epochs 5 --lr 4e-6 --alphas 0.7 --init twosense",
    "python src/train.py --tag f2sig2 --model ViT-SO400M-14-SigLIP2 --pretrained webli --epochs 6 --lr 4e-6 --alphas 0.7",
    "python src/dinotrain.py --tag d1dino --model vit_large_patch14_reg4_dinov2.lvd142m --epochs 7 --lr 1e-5",
    "python src/inettrain.py --tag g1eva448 --model eva02_large_patch14_448.mim_m38m_ft_in22k_in1k --res 448 --epochs 4 --lr 5e-6",
]

pd.DataFrame(
    [{"member": name, "weight": weight} for name, weight in MEMBERS.items()]
)

,member,weight
0,f1peL_a07,2
1,k2meta_a07,2
2,v3metaT_a07,2
3,d1dino_a10,3
4,g1eva448_a07,2
5,f2sig2_a07,1


## 4. Run Training or Fine-Tuning

Set `RUN_TRAINING = True` only in the supplied CUDA environment with the dataset present. The command streams epoch-level output from the trainer and writes checkpoints and probability arrays.

In [4]:
RUN_TRAINING = False

if RUN_TRAINING:
    if not DATA_AVAILABLE:
        raise FileNotFoundError("Training data is required for fine-tuning.")
    if DEVICE != "cuda":
        raise RuntimeError("A CUDA GPU is required for the submitted training configuration.")

    command = TRAINING_COMMANDS[0]
    print(f"running: {command}")
    started = time.time()
    process = subprocess.Popen(
        command.split(), cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
    if process.wait() != 0:
        raise RuntimeError(f"training failed: {command}")
    print(f"elapsed seconds: {time.time() - started:.0f}")
else:
    print("Training command prepared. Set RUN_TRAINING=True in a CUDA environment to run it.")

Training command prepared. Set RUN_TRAINING=True in a CUDA environment to run it.


## 5. Capture Training Traces and Metrics

The included arrays are outputs of the fine-tuned members. This cell produces an executed validation trace for every stored member.

In [5]:
trace_rows = []
for member, weight in MEMBERS.items():
    arrays = np.load(PROBS_DIR / f"probs_{member}.npz")
    validation_accuracy = (arrays["val"].argmax(axis=1) == arrays["yv"]).mean()
    trace_rows.append(
        {
            "member": member,
            "weight": weight,
            "validation_accuracy": validation_accuracy,
            "validation_rows": len(arrays["yv"]),
            "test_rows": len(arrays["test"]),
        }
    )

trace = pd.DataFrame(trace_rows).sort_values("validation_accuracy", ascending=False)
print("Stored fine-tuning trace")
print(trace.to_string(index=False, formatters={"validation_accuracy": "{:.4%}".format}))
trace

Stored fine-tuning trace
      member  weight validation_accuracy  validation_rows  test_rows
  k2meta_a07       2            99.5776%              947      11681
 v3metaT_a07       2            99.5776%              947      11681
g1eva448_a07       2            99.5776%              947      11681
  f2sig2_a07       1            99.5776%              947      11681
   f1peL_a07       2            99.4720%              947      11681
  d1dino_a10       3            99.2608%              947      11681


,member,weight,validation_accuracy,validation_rows,test_rows
1,k2meta_a07,2,0.995776,947,11681
2,v3metaT_a07,2,0.995776,947,11681
4,g1eva448_a07,2,0.995776,947,11681
5,f2sig2_a07,1,0.995776,947,11681
0,f1peL_a07,2,0.994720,947,11681
3,d1dino_a10,3,0.992608,947,11681


## 6. Evaluate the Trained Model

The weighted blend is evaluated on the held-out validation rows stored with each member. The same weights generate the final submission.

In [6]:
blend_validation = None
labels = None
for member, weight in MEMBERS.items():
    arrays = np.load(PROBS_DIR / f"probs_{member}.npz")
    blend_validation = arrays["val"] * weight if blend_validation is None else blend_validation + arrays["val"] * weight
    labels = arrays["yv"]

blend_validation /= sum(MEMBERS.values())
ensemble_accuracy = (blend_validation.argmax(axis=1) == labels).mean()
print(f"weighted ensemble validation accuracy: {ensemble_accuracy:.4%} ({int((blend_validation.argmax(axis=1) == labels).sum())}/{len(labels)})")
pd.Series(blend_validation.argmax(axis=1)).value_counts().sort_index().rename("validation_predictions")

weighted ensemble validation accuracy: 99.6832% (944/947)


0     50
1     33
2     47
3     52
4     58
5     50
6     54
7     57
8     48
9     45
10    52
11    46
12    45
13    49
14    38
15    31
16    52
17    55
18    51
19    34
Name: validation_predictions, dtype: int64

## 7. Save Model Artifacts

The submission uses the stored probability arrays rather than serializing the large backbone checkpoints. This cell records the measured trace and rebuilds the final CSV.

In [7]:
OUTPUT_DIR.mkdir(exist_ok=True)
trace_path = OUTPUT_DIR / "stored_finetuning_trace.csv"
trace.to_csv(trace_path, index=False)

rebuilt_path = OUTPUT_DIR / "rebuilt_submission.csv"
completed = subprocess.run(
    [sys.executable, "src/predict.py", "--probs", "probs", "--out", str(rebuilt_path)],
    cwd=ROOT,
    check=True,
    text=True,
    capture_output=True,
)
print(completed.stdout, end="")
print(f"saved trace: {trace_path}")
print(f"submission matches: {rebuilt_path.read_bytes() == (ROOT / 'submission.csv').read_bytes()}")

  f1peL_a07        weight 2  shape (11681, 20)
  k2meta_a07       weight 2  shape (11681, 20)
  v3metaT_a07      weight 2  shape (11681, 20)
  d1dino_a10       weight 3  shape (11681, 20)
  g1eva448_a07     weight 2  shape (11681, 20)
  f2sig2_a07       weight 1  shape (11681, 20)
wrote /Users/harveyhoulahan/Desktop/Projects/maincode/kagglerun/33970076_assignment01_report/notebook_outputs/rebuilt_submission.csv: 11681 rows, 20 classes
saved trace: /Users/harveyhoulahan/Desktop/Projects/maincode/kagglerun/33970076_assignment01_report/notebook_outputs/stored_finetuning_trace.csv
submission matches: True


## 8. Export the Notebook to HTML and PDF

Run the export command after executing the notebook. The submitted HTML export must include the stored validation trace, ensemble evaluation, and reconstruction check.

In [8]:
NOTEBOOK = ROOT / "assignment_notebook.ipynb"
HTML_EXPORT = ROOT / "assignment_notebook.html"
PDF_EXPORT = ROOT / "assignment_notebook.pdf"

print("Export after execution:")
print(f"{sys.executable} -m jupyter nbconvert --to html --execute --output {HTML_EXPORT.name} {NOTEBOOK.name}")
print(f"{sys.executable} -m jupyter nbconvert --to webpdf --output {PDF_EXPORT.name} {NOTEBOOK.name}")

Export after execution:
/Users/harveyhoulahan/Desktop/Projects/maincode/kagglerun/33970076_assignment01_report/.venv/bin/python -m jupyter nbconvert --to html --execute --output assignment_notebook.html assignment_notebook.ipynb
/Users/harveyhoulahan/Desktop/Projects/maincode/kagglerun/33970076_assignment01_report/.venv/bin/python -m jupyter nbconvert --to webpdf --output assignment_notebook.pdf assignment_notebook.ipynb
